# Stage 2/3 — Colab T4 Execution & Resume Layer

This notebook provides a 100% self-contained, auto-resuming execution pipeline for Google Colab.

### Key Features:
1. **Zero Manual File Uploads**: Clones and updates the repository directly from GitHub (`https://github.com/aliakarma/Safe-Lie.git`).
2. **Google Drive Integration**: All checkpoints (`checkpoint.pt`) and JSONL logs (`rounds.jsonl`, `oracle.jsonl`) are written directly to Google Drive.
3. **Automatic Resumption**: If a Colab session disconnects, runs out of memory, or timeouts, re-running this notebook automatically detects the checkpoint in Drive and seamlessly continues from the exact round without losing progress.
4. **Deterministic**: Bitwise-identical RNG and state continuation verified by test suite.

## Cell 1 — Mount Google Drive

Mounts Google Drive to `/content/drive` so all training progress and checkpoints persist permanently.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully at /content/drive")
except ImportError:
    print("Not running in Google Colab environment -- running locally.")

## Cell 2 — Clone Repository from GitHub (No manual uploads)

Clones `Safe-Lie` from GitHub or pulls latest changes if already present, and enters the repository directory.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/aliakarma/Safe-Lie.git"
REPO_DIR = "/content/Safe-Lie"

if os.path.exists("/content") and not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} into {REPO_DIR}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.exists(REPO_DIR):
    print(f"Updating existing repo at {REPO_DIR} via git pull...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    os.chdir(REPO_DIR)

print(f"Working directory: {os.getcwd()}")

## Cell 3 — Runtime & GPU Verification

Checks CUDA availability and device specifications.

In [ ]:
import os
import platform
import torch

print(f"Python version: {platform.python_version()}")
print(f"Platform: {platform.platform()}")
print(f"CPU count: {os.cpu_count()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Cell 4 — Dependency Installation

Installs pinned dependencies and the editable `safelie` package.

In [ ]:
!pip install -r requirements.txt
!pip install -e .

## Cell 5 — Version Verification & Preflight Gate

Runs the fast preflight test gate to verify all modules and deterministic seeds before launching experiments.

In [ ]:
import numpy
import pydantic
import scipy
import safelie
from safelie.preflight import run_preflight

print(f"safelie: {safelie.__version__}")
print(f"numpy: {numpy.__version__}")
print(f"scipy: {scipy.__version__}")
print(f"pydantic: {pydantic.VERSION}")
print(f"torch: {torch.__version__}")

preflight_exit_code = run_preflight(fast=True)
assert preflight_exit_code == 0, "Preflight gate failed -- do not proceed to training."
print("✓ Preflight passed cleanly.")

## Cell 6 — Config Selection & Google Drive Output Directory

Selects the experiment configuration and routes outputs directly to Google Drive for persistence across sessions.

In [ ]:
import os
from pathlib import Path
from safelie.utils.config import load_experiment_config

# Choose experiment: e.g. pilot_A_clean, pilot_B_attack, pilot_C_rce, pilot_D_benign, pilot_E_clean_rce, or local_demo_clean
EXPERIMENT_NAME = "pilot_A_clean"
cfg = load_experiment_config(f"configs/experiment/{EXPERIMENT_NAME}.yaml")

# Direct checkpoint storage to Google Drive if mounted
if os.path.exists("/content/drive/MyDrive"):
    DRIVE_RESULTS_DIR = "/content/drive/MyDrive/safelie_results"
    cfg = cfg.model_copy(update={"output_dir": DRIVE_RESULTS_DIR})
    print(f"Output directory routed to Google Drive: {Path(cfg.output_dir) / cfg.run_id}")
else:
    print(f"Output directory (local): {Path(cfg.output_dir) / cfg.run_id}")

print(f"Selected: {cfg.run_id} | Env: {cfg.env.name} | Steps: {cfg.total_steps} | Attack: {cfg.attack.name} | Defense: {cfg.defense.name}")

## Cell 7 — Checkpoint Inspection

Checks whether a previous checkpoint exists in Google Drive.

In [ ]:
from pathlib import Path

ckpt_file = Path(cfg.output_dir) / cfg.run_id / "checkpoint.pt"
if ckpt_file.exists():
    print(f"✓ Existing checkpoint found at: {ckpt_file}")
    print("Training in Cell 8 will automatically resume from the last completed round!")
else:
    print(f"No existing checkpoint found at {ckpt_file}. Will start fresh from round 0.")

## Cell 8 — Run Training with Auto-Checkpoint & Auto-Resume

Executes the experiment. Saves `checkpoint.pt` every round directly to Drive. If interrupted, re-running this cell automatically picks up from the exact round.

In [ ]:
from safelie.experiment import run_experiment_with_oracle

if cfg.env.name == "synthetic_constrained_marl":
    out_dir = run_experiment_with_oracle(
        cfg,
        eval_every=1,
        checkpoint_every=1,
        auto_resume=True,
    )
    print(f"Run complete/up-to-date at: {out_dir}")
else:
    try:
        out_dir = run_experiment_with_oracle(
            cfg,
            eval_every=1,
            checkpoint_every=1,
            auto_resume=True,
        )
        print(f"Run complete/up-to-date at: {out_dir}")
    except NotImplementedError as exc:
        print(f"Environment adapter needed for {cfg.env.name}: {exc}")

## Cell 9 — Analysis & Summary Table

Parses the generated `rounds.jsonl` and `oracle.jsonl` files from Drive and builds the results table.

In [ ]:
from pathlib import Path
from safelie.analysis.tables import build_summary_table

run_dir = Path(cfg.output_dir) / cfg.run_id
if run_dir.exists():
    print(build_summary_table({cfg.run_id: run_dir}, budget=cfg.env.budget))
else:
    print(f"No completed run found at {run_dir} yet.")